# Collect Data

# Prepare data
## Data preparation - step 1



In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.functions import col
from pyspark.sql.functions import col, isnan

In the following, knowing that there is no header in the files and ";" is the seperator, we specified those options to import and merge the prices files from 2022 to 2024.

In [35]:
spark = (SparkSession.builder
         .appName("Prix")
         .getOrCreate())

files_path = "../Data/1-raw/Prix*.csv.gz"

df_prices = (spark.read
    #.option("header", True)
    .option("inferSchema", True) 
    .option("sep", ";")              
    .csv(files_path)                 
)

df_prices.head(5)

[Row(_c0=1000001, _c1=1000, _c2='R', _c3=4620100.0, _c4=519800.0, _c5=datetime.datetime(2023, 1, 2, 7, 53, 26), _c6=1, _c7='Gazole', _c8=1.867),
 Row(_c0=1000001, _c1=1000, _c2='R', _c3=4620100.0, _c4=519800.0, _c5=datetime.datetime(2023, 1, 5, 9, 33, 37), _c6=1, _c7='Gazole', _c8=1.877),
 Row(_c0=1000001, _c1=1000, _c2='R', _c3=4620100.0, _c4=519800.0, _c5=datetime.datetime(2023, 1, 9, 14, 51, 49), _c6=1, _c7='Gazole', _c8=1.875),
 Row(_c0=1000001, _c1=1000, _c2='R', _c3=4620100.0, _c4=519800.0, _c5=datetime.datetime(2023, 1, 11, 9, 23, 54), _c6=1, _c7='Gazole', _c8=1.859),
 Row(_c0=1000001, _c1=1000, _c2='R', _c3=4620100.0, _c4=519800.0, _c5=datetime.datetime(2023, 1, 13, 9, 7, 40), _c6=1, _c7='Gazole', _c8=1.862)]

In the read.me, we saw the name of the columns and their descriptions. We just included them in order to rename our columns.

In [36]:
cols = [
    "id_pdv", "cp", "pop",
    "latitude", "longitude",
    "date",
    "id_carburant", "nom_carburant",
    "prix"
]

df_prices = df_prices.toDF(*cols)
df_prices.show(5)
df_prices.printSchema()

+-------+----+---+---------+---------+-------------------+------------+-------------+-----+
| id_pdv|  cp|pop| latitude|longitude|               date|id_carburant|nom_carburant| prix|
+-------+----+---+---------+---------+-------------------+------------+-------------+-----+
|1000001|1000|  R|4620100.0| 519800.0|2023-01-02 07:53:26|           1|       Gazole|1.867|
|1000001|1000|  R|4620100.0| 519800.0|2023-01-05 09:33:37|           1|       Gazole|1.877|
|1000001|1000|  R|4620100.0| 519800.0|2023-01-09 14:51:49|           1|       Gazole|1.875|
|1000001|1000|  R|4620100.0| 519800.0|2023-01-11 09:23:54|           1|       Gazole|1.859|
|1000001|1000|  R|4620100.0| 519800.0|2023-01-13 09:07:40|           1|       Gazole|1.862|
+-------+----+---+---------+---------+-------------------+------------+-------------+-----+
only showing top 5 rows

root
 |-- id_pdv: integer (nullable = true)
 |-- cp: integer (nullable = true)
 |-- pop: string (nullable = true)
 |-- latitude: double (nullable =

Knowing the previous format of the files, we use the withColumn methods to extract the year, the month and the week of the year.

In [37]:
df_prices = (df_prices
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("week_of_year", F.weekofyear("date"))
)

df_prices.select("date","year","month","week_of_year").show(5)

+-------------------+----+-----+------------+
|               date|year|month|week_of_year|
+-------------------+----+-----+------------+
|2023-01-02 07:53:26|2023|    1|           1|
|2023-01-05 09:33:37|2023|    1|           1|
|2023-01-09 14:51:49|2023|    1|           2|
|2023-01-11 09:23:54|2023|    1|           2|
|2023-01-13 09:07:40|2023|    1|           2|
+-------------------+----+-----+------------+
only showing top 5 rows



In [39]:
# We divide the columns longitude and latitude by 10^5
# as it was making sense for the first one and keeping five zeros is common in order to get a 1-meter precision 
df_prices = (df_prices 
.withColumn("latitude", (F.col("latitude") / 100000)) 
.withColumn("longitude", (F.col("longitude") / 100000))
)

df_prices.select("latitude", "longitude").show(5)




+--------+---------+
|latitude|longitude|
+--------+---------+
|  46.201|    5.198|
|  46.201|    5.198|
|  46.201|    5.198|
|  46.201|    5.198|
|  46.201|    5.198|
+--------+---------+
only showing top 5 rows



In [ ]:
# We try to identify all the rows with missing values
# As they are a few and they correspond to a small proportion, we will drop them all
condition = None
for c in df_prices.columns:
    if condition is None:
        condition = col(c).isNull() 
    else:
        condition = condition | col(c).isNull()

rows_with_na = df_prices.filter(condition)


count_rows_with_na = rows_with_na.count()
print(f"Numbers of rows with at least one NA: {count_rows_with_na}")
print(f"Proportion of rows with at least one NA: {count_rows_with_na / df_prices.count()}")


Numbers of rows with at least one NA: 12756


Proportion of rows with at least one NA: 0.000897372231563401


In [44]:
df_prices_filtered = df_prices.na.drop()

In [ ]:
# In order to make the table available to spark SQL we use the the createOrReplaceTempView method
# We rename it prices_SQL
df_prices_filtered.createOrReplaceTempView('prices_SQL')
n_data = df_prices_filtered.count()

# We now will try to the distribution of each gas type in order to choose the two out of our interest
spark.sql(f""" 
SELECT nom_carburant, 
COUNT(*) * 1.0 / {n_data} AS freq 
FROM prices_SQL 
GROUP BY nom_carburant 
ORDER BY freq DESC 
"""
).show()


spark.sql(f"""
SELECT
  year,
  nom_carburant,
  COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY year) AS freq
FROM prices_SQL
GROUP BY year, nom_carburant
ORDER BY year, freq DESC
""").show()


We notice by computing the proportion of each type of Gas by year and overall that the gas types SP95 and GPLc are the least frequent with less than 7% each year so we decide to 

# Visualize gas prices

# Model gas prices evolution

# Evaluate if electric cars development as an impact